In [1]:
#| default_exp machine_learning.producers.note_division

> **AI-generated — pending human review.** Created by Claude Code on 2026-10-22. Review checklist: `_nbdev_development/AI_REVIEW_CHECKLIST.md` (workspace root). Remove this cell only after review passes.

Second producer of the R2/R9 family: the dataset contract for pipeline H (note division), reformulated per R8 from fastai windows to token classification. Mapping old (`trouver_plus/markdown/obsidian/personal/machine_learning/note_splitting.py` + playground `ml_data_update_note_division.ipynb`) → new:

| Old | New |
|---|---|
| 5-line windows over preprocessed notes, one binary fastai decision per window (O(windows) inference; `lines_in_point=5`, `split_at=2`) | token classification over a joined sequence: boundaries are zero-width `split` spans at chunk seams, one batched pass |
| label = *presence of a file boundary* between consecutive index-ordered sibling notes (every seam `True`, every intra-note juncture `False`; no tags, block IDs, or pre/post alignment involved) | same ground truth re-expressed as seam spans — **its derivation stays in trouver_plus** (personal-vault provenance: which consecutive notes form one divided original); this module owns only the row shape |
| `note_division.csv` (date columns, pandas dedup; cannot run today — `df.append` was removed in pandas ≥ 2.0) | schema-validated JSONL via `ml_data`, like `def_and_notat_ner_v2` |

This module is deliberately vault-agnostic: no imports from the tokenize chain or `VaultNote`, stdlib + `ml_data` only, so it stays importable anywhere in the contract layer. Sequencing (label derivation), training, and the trained artifact live outside this repo; no `MODEL_IDS` entry is registered for this task.

In [2]:
#| export
from collections.abc import Iterable
from os import PathLike
from pathlib import Path

from trouver.machine_learning.ml_data import (PROVENANCE_COLUMN, DatasetSchema, dataset_dir,
                                              save_rows)

NOTE_DIVISION_SCHEMA = DatasetSchema('note_division', 1, ('task', 'Note name', 'text', 'spans'))
"""Canonical schema of the pure H dataset: `spans` is a list of zero-width
`{'start', 'end', 'label'}` dicts with `start == end` and `label == 'split'`,
char offsets into `text` — each span marks a boundary where a human actually
divided one note into the consecutive notes whose texts were joined.
Tokenizer-independent by design, like `DEF_AND_NOTAT_NER_SCHEMA` (`seam_spans`)."""

"Canonical schema of the pure H dataset: `spans` is a list of zero-width\n`{'start', 'end', 'label'}` dicts with `start == end` and `label == 'split'`,\nchar offsets into `text` — each span marks a boundary where a human actually\ndivided one note into the consecutive notes whose texts were joined.\nTokenizer-independent by design, like `DEF_AND_NOTAT_NER_SCHEMA` (`seam_spans`)."

In [3]:
#| export
def seam_spans(chunks: list[str], joiner: str = '\n') -> list[dict]:
    """
    Zero-width `split` boundary spans at the seams between ordered `chunks`.

    A chunk sequence is one already-divided original (the consecutive notes a
    human split it into), so every seam is a positive label and no extra
    annotation exists: negatives are precisely the positions of `text` that
    carry no span. Offsets count into the text built by `note_division_row`.

    **Parameters**
    - `chunks` (*list[str]*) — Ordered chunk texts, as joined by `note_division_row`.
    - `joiner` (*str*) — String placed between chunks when joining; seam offsets account for it.
    """
    spans, pos = [], 0
    for chunk in chunks[:-1]:
        pos += len(chunk)
        spans.append({'start': pos, 'end': pos, 'label': 'split'})
        pos += len(joiner)
    return spans


def note_division_row(note_name: str, chunks: list[str], joiner: str = '\n') -> dict:
    """
    One `ml_data` row for an ordered chunk sequence: joined `text` plus its seam spans.

    **Parameters**
    - `note_name` (*str*) — Provenance identifier (usually the parent note's name).
    - `chunks` (*list[str]*) — Ordered texts of the consecutive divided notes.
    - `joiner` (*str*) — Separator used to build `text`; must match `seam_spans`.
    """
    return {'task': 'note_division', PROVENANCE_COLUMN: note_name,
            'text': joiner.join(chunks), 'spans': seam_spans(chunks, joiner)}

In [4]:
#| export
def produce_note_division_rows(sequences: Iterable[tuple[str, list[str]]],
                               joiner: str = '\n') -> list[dict]:
    """
    Turn `(note_name, chunks)` sequences into validated note-division rows; saved nowhere.

    Which consecutive notes form one divided original is the caller's decision
    (label derivation from vault provenance lives in trouver_plus); this only
    shapes and validates rows — no augmentation (R5), no splitting
    (`train_split` owns it), no writing. A single-chunk sequence yields a
    legitimate all-negative row, mirroring `def_and_notat_ner`'s negatives.

    **Parameters**
    - `sequences` (*Iterable[tuple[str, list[str]]]*) — `(note_name, ordered chunk texts)` pairs.
    - `joiner` (*str*) — Separator between chunks; must match training-time tokenization of `text`.
    """
    rows = [note_division_row(name, chunks, joiner) for name, chunks in sequences]
    NOTE_DIVISION_SCHEMA.validate_rows(rows)
    return rows


def save_note_division_dataset(rows: list[dict], root: PathLike) -> Path:
    """
    Write `rows` (from `produce_note_division_rows`) as the canonical JSONL dataset; return its path.

    **Parameters**
    - `rows` (*list[dict]*) — Produced rows.
    - `root` (*PathLike*) — Data root (e.g. `Python/trouver_data`); writes `_ml_data/note_division_v1/note_division.jsonl`.
    """
    path = dataset_dir(root, NOTE_DIVISION_SCHEMA) / 'note_division.jsonl'
    save_rows(rows, path)
    return path

In [5]:
#| hide
import json
import tempfile
from pathlib import Path

from fastcore.test import *

from trouver.machine_learning import ml_data as mld
from trouver.machine_learning.producers import note_division as prod

In [6]:
#| hide
# Seam offsets: zero-width spans after each chunk (+joiner); single/empty chunk = no seams.
chunks = ['ab', 'cde', 'f']
test_eq(prod.seam_spans(chunks), [{'start': 2, 'end': 2, 'label': 'split'},
                                  {'start': 6, 'end': 6, 'label': 'split'}])
test_eq(prod.seam_spans(['only']), [])
test_eq(prod.seam_spans([]), [])
row = prod.note_division_row('n', chunks)
test_eq(row['text'], 'ab\ncde\nf')
for s in row['spans']:  # zero-width, and the seam position is the joiner itself
    test_eq(s['start'], s['end'])
    test_eq(row['text'][s['start']], '\n')
test_eq(prod.seam_spans(chunks, '||'), [{'start': 2, 'end': 2, 'label': 'split'},
                                        {'start': 7, 'end': 7, 'label': 'split'}])

In [7]:
#| hide
# Row contract: task/provenance columns, JSON safety, schema validation, negatives kept.
json.loads(json.dumps([row]))
test_eq(row['task'], 'note_division')
test_eq(row[mld.PROVENANCE_COLUMN], 'n')
neg = prod.note_division_row('m', ['solo'])
test_eq(neg['spans'], [])
prod.NOTE_DIVISION_SCHEMA.validate_rows([row, neg])

# produce: shapes + validates + keeps order; save/load round-trips as pure.
rows = prod.produce_note_division_rows([('n', chunks), ('m', ['solo'])])
test_eq([r[mld.PROVENANCE_COLUMN] for r in rows], ['n', 'm'])
with tempfile.TemporaryDirectory() as tmp:
    path = prod.save_note_division_dataset(rows, Path(tmp))
    assert path.exists() and path.parent.name == 'note_division_v1'
    back = mld.load_rows(path, schema=prod.NOTE_DIVISION_SCHEMA, require_pure=True)
    test_eq(back, rows)

## Examples

```python
# Sequencing comes from label derivation in trouver_plus: consecutive sibling
# notes in index order are the chunks of one divided original.
sequences = [(parent.name, [text_of(n) for n in run])]  # run = [parent, child_2, ...]
rows = produce_note_division_rows(sequences)
path = save_note_division_dataset(rows, '<data root>')
# -> <root>/_ml_data/note_division_v1/note_division.jsonl (pure rows, schema-validated)
```